# Janelas mensais GARE — EDA granular e baselines leves (v0 expand)

**Monthly GARE windows — granular EDA and light baselines (v0, expanded)**

## Resumo

Complementa o painel mensal 119m (`estoque_arrecadacao_eda_forecast_v0.ipynb`) com **8–12 janelas de um mês civil** dentro de `extracao=2026-03`. Preferência: **Parquet cache local** sob `${DUMP_ROOT}/extracao=…/samples/`; API `/v1/data/arrecadacao` só em cache-miss, com páginas curtas, timeout moderado, retries com backoff e flag `partial=True` se interromper. **Não** chamamos `/v1/data/debito` row-level (já ReadTimeout) — join só contra dump local de débito. Grão analítico: **agregado diário** de `VALOR_TOTAL_GARE`. Sem PII. Conteúdo e conclusões do autor; formatação com assistência de IA.

## Palavras-chave

GARE; janela mensal; arrecadação; walk-forward; sample API; parquet cache; timeout-safe


## 1. Critério das janelas (documentado)

Escolha **dentro** do período coberto pelo painel da extração `2026-03` (série mensal 2016-01→2026-03). Mix pedido: típico / alta arrecadação / outlier estoque / pico sazonal / mid / recente-mas-não-último.

| ID | Critério | Regra de seleção |
|---|---|---|
| W1 | Típico | `valor_total` mais próximo da mediana do painel |
| W2 | Outlier estoque | Máximo `valor_sem_honorarios` |
| W3 | Alta arrecadação | Máximo `valor_total` |
| W4 | Pico sazonal | Mês-calendário com maior média histórica de `valor_total`; ano mais recente |
| W5 | Mid série | Ponto médio temporal do painel |
| W6 | Recente (não último) | Penúltimo mês (evita foto incompleta) |
| W7 | Sazonalidade baixa | Mês-calendário com menor média histórica; ano mais recente |
| W8 | Alta arrec. recente | Máximo `valor_total` nos últimos 24 meses |
| W9 | Outlier estoque (IQR) | Segundo extremo acima de Q3+1.5·IQR (se o max já for W2) |
| W10 | Típico alternativo | Segundo mês mais próximo da mediana (ano distinto) |
| W11 | P75 arrecadação | Mês com `valor_total` ≈ percentil 75 |
| W12 | Típico período inicial | Mediana-like no primeiro terço da série |

**Budget de execução:** preferir **≥6 janelas full** (EDA + agregado diário + baselines) quando cache/API permitir; restantes em modo *light* (n / missingness / valor_sum). Débito: só cache local.


## 2. Setup (`.env`, cliente, higiene de paths)


In [1]:
from pathlib import Path
import os, sys, warnings, time, math
warnings.filterwarnings("ignore")

NB_DIR = Path.cwd()
candidates = [NB_DIR, *NB_DIR.parents]
MONO = next((p for p in candidates if (p / "shared" / "cemepi_api").exists()), NB_DIR)
os.chdir(MONO); sys.path.insert(0, str(MONO))

from shared.cemepi_api import CemepiClient, load_settings
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

FIG_DIR = MONO / "projects/monitoramento/output/figures/gare"
FIG_DIR.mkdir(parents=True, exist_ok=True)

S = load_settings(); S.ano, S.mes = 2026, 3
C = CemepiClient(S)
EXTRACAO = f"{S.ano:04d}-{S.mes:02d}"
SEED = 42; np.random.seed(SEED)

# Timeout-safe loader knobs
API_TIMEOUT = 45          # modest raise vs 30s (debito still blocked)
PAGE_LIMIT = 200          # short pages
MAX_RETRIES = 3
BACKOFF_BASE_S = 1.5
MAX_PAGES_FULL = 10       # ≤2000 rows full
MAX_PAGES_LIGHT = 3       # ≤600 rows light
N_FULL_TARGET = 6         # at least 6 full EDA+forecast when data allows
TIMEOUT_INCIDENTS = []    # audit log

def public_path_label(p: Path) -> str:
    s = str(p)
    for key in ("DUMP_ROOT", "LAKE_ROOT", "COLLECT_ROOT"):
        root = os.getenv(key)
        if root and s.startswith(root):
            return f"${{{key}}}/" + s[len(root):].lstrip("/")
    if s.startswith(str(MONO)):
        return "mono:/" + s[len(str(MONO)):].lstrip("/")
    return p.name

def save_fig(fig, stem: str, width: int = 1000, height: int = 520) -> Path:
    out = FIG_DIR / f"{stem}.png"
    try:
        fig.write_image(str(out), width=width, height=height, scale=2)
        print("static ->", public_path_label(out))
    except Exception as e:
        print("Kaleido write_image skip:", type(e).__name__, e)
    return out

PII_COLS = {"NOME_DEVEDOR", "CPF_DEVEDOR", "CNPJ_DEVEDOR", "NOME", "CPF", "CNPJ"}

def drop_pii(df: pd.DataFrame) -> pd.DataFrame:
    cols = [c for c in df.columns if c.upper() in PII_COLS or any(x in c.upper() for x in ["CPF", "NOME_DEV", "CNPJ"])]
    return df.drop(columns=cols, errors="ignore")

def normalize_gare(df: pd.DataFrame) -> pd.DataFrame:
    df = drop_pii(df).copy()
    df.columns = [c.upper() for c in df.columns]
    if "DATA_ARRECADACAO_GARE" in df.columns:
        df["DATA_ARRECADACAO_GARE"] = pd.to_datetime(df["DATA_ARRECADACAO_GARE"], errors="coerce")
    for c in ["VALOR_TOTAL_GARE", "VALOR_RECEITA", "JUROS_MORA", "MULTA_MORA", "ID_DEBITO"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

print("extracao_ref", EXTRACAO, "| sample_dir", public_path_label(C.sample_dir()))
print("FIG_DIR", public_path_label(FIG_DIR))
print("knobs", dict(API_TIMEOUT=API_TIMEOUT, PAGE_LIMIT=PAGE_LIMIT, MAX_PAGES_FULL=MAX_PAGES_FULL,
                    MAX_PAGES_LIGHT=MAX_PAGES_LIGHT, N_FULL_TARGET=N_FULL_TARGET, MAX_RETRIES=MAX_RETRIES))


extracao_ref 2026-03 | sample_dir ${DUMP_ROOT}/extracao=2026-03/samples
FIG_DIR mono:/projects/monitoramento/output/figures/gare
knobs {'API_TIMEOUT': 45, 'PAGE_LIMIT': 200, 'MAX_PAGES_FULL': 10, 'MAX_PAGES_LIGHT': 3, 'N_FULL_TARGET': 6, 'MAX_RETRIES': 3}


## 3. Painel mensal local e seleção das 12 janelas

Lemos `painel_mensal_estoque_arrecadacao.parquet` (paths públicos via `public_path_label`).


In [2]:
painel_path = C.sample_dir() / "painel_mensal_estoque_arrecadacao.parquet"
panel = pd.read_parquet(painel_path)
panel["data"] = pd.to_datetime(panel["data"])
panel = panel.sort_values("data").reset_index(drop=True)
print("painel", public_path_label(painel_path), "| n", len(panel),
      "| range", panel["data"].min().date(), "→", panel["data"].max().date())

med = float(panel["valor_total"].median())
seas = panel.groupby("mes")["valor_total"].mean()
hot_month = int(seas.idxmax())
cold_month = int(seas.idxmin())
q1, q3 = panel["valor_sem_honorarios"].quantile([0.25, 0.75])
fence = float(q3 + 1.5 * (q3 - q1))
p75 = float(panel["valor_total"].quantile(0.75))

used = set()
rows = []

def take(row, criterio: str):
    key = (int(row.ano), int(row.mes))
    if key in used:
        return False
    used.add(key)
    rows.append({
        "criterio": criterio,
        "ano": key[0], "mes": key[1],
        "valor_total": float(row.valor_total),
        "estoque": float(row.valor_sem_honorarios),
    })
    return True

take(panel.iloc[(panel["valor_total"] - med).abs().argmin()], "tipico_mediana_valor_total")
take(panel.loc[panel["valor_sem_honorarios"].idxmax()], "outlier_max_estoque")
take(panel.loc[panel["valor_total"].idxmax()], "alta_arrecadacao_max")
take(panel[panel["mes"] == hot_month].iloc[-1], f"seasonal_peak_mes_{hot_month}")
take(panel.iloc[len(panel) // 2], "mid_serie_temporal")
take(panel.iloc[-2], "recente_penultimo")
take(panel[panel["mes"] == cold_month].iloc[-1], f"seasonal_low_mes_{cold_month}")
take(panel.iloc[-24:].loc[panel.iloc[-24:]["valor_total"].idxmax()], "alta_arrec_recente_24m")

out_iqr = panel[panel["valor_sem_honorarios"] > fence].sort_values("valor_sem_honorarios", ascending=False)
for _, r in out_iqr.iterrows():
    if take(r, "outlier_estoque_iqr_upper"):
        break

rest = panel.copy()
rest["_k"] = list(zip(rest.ano.astype(int), rest.mes.astype(int)))
rest = rest[~rest["_k"].isin(used)]
take(rest.iloc[(rest["valor_total"] - med).abs().argmin()], "tipico_alt_mediana")
rest = rest[~rest.apply(lambda r: (int(r.ano), int(r.mes)) in used, axis=1)]
take(rest.iloc[(rest["valor_total"] - p75).abs().argmin()], "p75_arrecadacao")
early = panel.iloc[: len(panel) // 3].copy()
early["_k"] = list(zip(early.ano.astype(int), early.mes.astype(int)))
early = early[~early["_k"].isin(used)]
if len(early):
    take(early.iloc[(early["valor_total"] - med).abs().argmin()], "tipico_periodo_inicial")

windows = pd.DataFrame(rows)
windows.insert(0, "id", [f"W{i+1}" for i in range(len(windows))])
assert 8 <= len(windows) <= 12, len(windows)
print("n_janelas", len(windows), "| hot_month", hot_month, "| cold_month", cold_month)
display(windows)


painel ${DUMP_ROOT}/extracao=2026-03/samples/painel_mensal_estoque_arrecadacao.parquet | n 119 | range 2016-01-01 → 2026-03-01
n_janelas 12 | hot_month 12 | cold_month 6


,id,criterio,ano,mes,valor_total,estoque
0,W1,tipico_mediana_valor_total,2018,1,2.863329e+08,3.736407e+11
1,W2,outlier_max_estoque,2020,6,1.829183e+08,6.716230e+11
2,W3,alta_arrecadacao_max,2019,12,1.729281e+09,3.217080e+11
3,W4,seasonal_peak_mes_12,2025,12,7.586049e+08,4.571766e+11
4,W5,mid_serie_temporal,2021,4,1.961604e+08,1.879849e+11
5,W6,recente_penultimo,2026,2,5.411049e+08,4.652432e+11
6,W7,seasonal_low_mes_6,2025,6,4.806183e+08,4.351448e+11
7,W8,alta_arrec_recente_24m,2024,4,9.657324e+08,2.516135e+11
8,W9,outlier_estoque_iqr_upper,2020,5,1.210767e+08,6.707873e+11
9,W10,tipico_alt_mediana,2017,3,2.880892e+08,3.471078e+11


## 4. Loader timeout-safe (cache → API parcial)

### Desenho

1. **Cache-first:** `${DUMP_ROOT}/extracao=…/samples/gare_window_{ano}_{mes:02d}.parquet` (também aceita sufixo `_sample` legado).
2. **API só em miss:** paginar `/v1/data/arrecadacao` com `ano`/`mes`, `limit=PAGE_LIMIT` curto, `timeout=API_TIMEOUT`, até `MAX_RETRIES` com backoff exponencial; em timeout/erro → **parar limpo**, manter páginas parciais, `partial=True`.
3. **Debito:** nunca row-level API; join só se existir Parquet local (`a3_debito_sample.parquet` etc.).


In [3]:
from requests.exceptions import ReadTimeout, ConnectTimeout, Timeout

def cache_paths(ano: int, mes: int) -> list[Path]:
    d = C.sample_dir()
    return [
        d / f"gare_window_{ano:04d}_{mes:02d}.parquet",
        d / f"gare_window_{ano:04d}_{mes:02d}_sample.parquet",
    ]

def fetch_arrec_window(ano: int, mes: int, max_pages: int, label: str) -> tuple[pd.DataFrame, bool]:
    # Paginate arrecadacao; return (df, partial). Never raises on timeout - stops cleanly.
    rows, cursor, pages = [], None, 0
    partial = False
    t0 = time.time()
    while pages < max_pages:
        params = {"limit": PAGE_LIMIT, "ano": int(ano), "mes": int(mes)}
        if cursor:
            params["cursor"] = cursor
        ok_payload = None
        last_err = None
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                r = C.session.get(
                    f"{C.s.base}/v1/data/arrecadacao",
                    params=params,
                    timeout=API_TIMEOUT,
                )
                r.raise_for_status()
                ok_payload = r.json()
                break
            except (ReadTimeout, ConnectTimeout, Timeout) as e:
                last_err = e
                TIMEOUT_INCIDENTS.append({
                    "label": label, "ano": ano, "mes": mes, "page": pages,
                    "attempt": attempt, "err": type(e).__name__,
                })
                sleep_s = BACKOFF_BASE_S * (2 ** (attempt - 1))
                print(f"{label} timeout page={pages} attempt={attempt}/{MAX_RETRIES}; backoff {sleep_s:.1f}s")
                time.sleep(sleep_s)
            except Exception as e:
                last_err = e
                TIMEOUT_INCIDENTS.append({
                    "label": label, "ano": ano, "mes": mes, "page": pages,
                    "attempt": attempt, "err": type(e).__name__,
                })
                print(f"{label} API error page={pages}:", type(e).__name__, e)
                break
        if ok_payload is None:
            partial = True
            print(f"{label} stop clean (partial) after page={pages}; last={type(last_err).__name__ if last_err else None}")
            break
        batch = ok_payload.get("dados") or ok_payload.get("data") or []
        if not batch:
            break
        rows.extend(batch)
        pages += 1
        cursor = (ok_payload.get("meta") or {}).get("proximo_cursor")
        if not cursor:
            break
    df = normalize_gare(pd.DataFrame(rows)) if rows else normalize_gare(pd.DataFrame())
    elapsed = round(time.time() - t0, 2)
    print(f"{label} ano={ano} mes={mes:02d} pages={pages} n={len(df)} partial={partial} elapsed_s={elapsed}")
    return df, partial

def load_or_fetch(ano: int, mes: int, max_pages: int, label: str) -> tuple[pd.DataFrame, str, bool]:
    for cache in cache_paths(ano, mes):
        if cache.exists():
            df = normalize_gare(pd.read_parquet(cache))
            print(label, "cache", public_path_label(cache), "n", len(df))
            return df, "parquet_cache", False
    df, partial = fetch_arrec_window(ano, mes, max_pages, label)
    # Prefer canonical name (no _sample) for new writes
    out = cache_paths(ano, mes)[0]
    if len(df):
        df.to_parquet(out, index=False)
        print(label, "saved", public_path_label(out), "partial", partial)
    return df, ("api_partial" if partial else "api_sample"), partial

# Probe which windows already have cache (drives full vs light priority)
cache_hits = []
for _, row in windows.iterrows():
    hit = any(p.exists() for p in cache_paths(int(row.ano), int(row.mes)))
    cache_hits.append(hit)
windows = windows.copy()
windows["has_cache"] = cache_hits
print("cache hits:", int(windows["has_cache"].sum()), "/", len(windows))
display(windows[["id", "ano", "mes", "criterio", "has_cache"]])


cache hits: 4 / 12


,id,ano,mes,criterio,has_cache
0,W1,2018,1,tipico_mediana_valor_total,True
1,W2,2020,6,outlier_max_estoque,True
2,W3,2019,12,alta_arrecadacao_max,False
3,W4,2025,12,seasonal_peak_mes_12,True
4,W5,2021,4,mid_serie_temporal,False
5,W6,2026,2,recente_penultimo,True
6,W7,2025,6,seasonal_low_mes_6,False
7,W8,2024,4,alta_arrec_recente_24m,False
8,W9,2020,5,outlier_estoque_iqr_upper,False
9,W10,2017,3,tipico_alt_mediana,False


## 5. Carregar todas as janelas + modo full vs light

Prioridade full: janelas com cache, depois as demais até `N_FULL_TARGET`. Light = EDA resumida sem forecast.


In [4]:
# Decide full set: prefer cache hits, fill to N_FULL_TARGET
full_ids = windows.loc[windows["has_cache"], "id"].tolist()
for wid in windows["id"]:
    if wid not in full_ids and len(full_ids) < N_FULL_TARGET:
        full_ids.append(wid)
FULL_IDS = set(full_ids)
print("FULL_IDS", sorted(FULL_IDS, key=lambda x: int(x[1:])))

loaded = {}  # id -> dict
for _, row in windows.iterrows():
    wid = row["id"]
    is_full = wid in FULL_IDS
    max_pages = MAX_PAGES_FULL if is_full else MAX_PAGES_LIGHT
    df, src, partial = load_or_fetch(int(row.ano), int(row.mes), max_pages, wid)
    loaded[wid] = {
        "df": df, "source": src, "partial": partial, "mode": "full" if is_full else "light",
        "ano": int(row.ano), "mes": int(row.mes), "criterio": row["criterio"],
    }
    print(f"  -> {wid} mode={loaded[wid]['mode']} n={len(df)} src={src} partial={partial}")

print("timeout_incidents_so_far", len(TIMEOUT_INCIDENTS))


FULL_IDS ['W1', 'W2', 'W3', 'W4', 'W5', 'W6']


W1 cache ${DUMP_ROOT}/extracao=2026-03/samples/gare_window_2018_01_sample.parquet n 4000
  -> W1 mode=full n=4000 src=parquet_cache partial=False
W2 cache ${DUMP_ROOT}/extracao=2026-03/samples/gare_window_2020_06_sample.parquet n 1000
  -> W2 mode=full n=1000 src=parquet_cache partial=False


W3 ano=2019 mes=12 pages=10 n=2000 partial=False elapsed_s=2.51
W3 saved ${DUMP_ROOT}/extracao=2026-03/samples/gare_window_2019_12.parquet partial False
  -> W3 mode=full n=2000 src=api_sample partial=False


W4 cache ${DUMP_ROOT}/extracao=2026-03/samples/gare_window_2025_12_sample.parquet n 1000
  -> W4 mode=full n=1000 src=parquet_cache partial=False


W5 ano=2021 mes=04 pages=10 n=2000 partial=False elapsed_s=2.29
W5 saved ${DUMP_ROOT}/extracao=2026-03/samples/gare_window_2021_04.parquet partial False
  -> W5 mode=full n=2000 src=api_sample partial=False
W6 cache ${DUMP_ROOT}/extracao=2026-03/samples/gare_window_2026_02_sample.parquet n 1000
  -> W6 mode=full n=1000 src=parquet_cache partial=False


...

W8 ano=2024 mes=04 pages=3 n=600 partial=False elapsed_s=1.57
W8 saved ${DUMP_ROOT}/extracao=2026-03/samples/gare_window_2024_04.parquet partial False
  -> W8 mode=light n=600 src=api_sample partial=False


W9 ano=2020 mes=05 pages=3 n=600 partial=False elapsed_s=1.19
W9 saved ${DUMP_ROOT}/extracao=2026-03/samples/gare_window_2020_05.parquet partial False
  -> W9 mode=light n=600 src=api_sample partial=False


W10 ano=2017 mes=03 pages=3 n=600 partial=False elapsed_s=1.29
W10 saved ${DUMP_ROOT}/extracao=2026-03/samples/gare_window_2017_03.parquet partial False
  -> W10 mode=light n=600 src=api_sample partial=False


W11 ano=2022 mes=01 pages=3 n=600 partial=False elapsed_s=1.28
W11 saved ${DUMP_ROOT}/extracao=2026-03/samples/gare_window_2022_01.parquet partial False
  -> W11 mode=light n=600 src=api_sample partial=False


W12 ano=2017 mes=05 pages=3 n=600 partial=False elapsed_s=0.63
W12 saved ${DUMP_ROOT}/extracao=2026-03/samples/gare_window_2017_05.parquet partial False
  -> W12 mode=light n=600 src=api_sample partial=False
timeout_incidents_so_far 0


## 6. Funções compartilhadas — diário, métricas, forecast leve

### Protocolo e métricas (primeira aparição — fórmulas)

#### Walk-forward / holdout

Teste = futuro relativo ao treino (Hyndman & Athanasopoulos, FPP3). Aqui: últimos `h = min(7, max(3, n//4))` dias do mês-calendário.

#### MAPE

$$\mathrm{MAPE} = \frac{1}{m}\sum_{t\in\mathcal{T}_{\mathrm{nz}}} \left|\frac{y_t - \hat y_t}{y_t}\right|$$

onde $\mathcal{T}_{\mathrm{nz}} = \{t: y_t \neq 0\}$ (dias com fluxo zero excluídos do MAPE).

#### SMAPE

$$\mathrm{SMAPE} = \frac{1}{m}\sum_t \frac{2\,|y_t - \hat y_t|}{|y_t| + |\hat y_t|}$$

(mais estável com zeros; FPP3 / Makridakis).

#### RMSE

$$\mathrm{RMSE} = \sqrt{\frac{1}{m}\sum_t (y_t - \hat y_t)^2}$$


In [5]:
def mape(a, f):
    a, f = np.asarray(a, float), np.asarray(f, float)
    m = a != 0
    return float(np.mean(np.abs((a[m] - f[m]) / a[m]))) if m.any() else np.nan

def smape(a, f):
    a, f = np.asarray(a, float), np.asarray(f, float)
    denom = np.abs(a) + np.abs(f)
    out = np.zeros_like(a, float)
    m = denom != 0
    out[m] = 2 * np.abs(a[m] - f[m]) / denom[m]
    return float(np.mean(out))

def rmse(a, f):
    return float(np.sqrt(np.mean((np.asarray(a, float) - np.asarray(f, float)) ** 2)))

def to_daily(df: pd.DataFrame, ano: int, mes: int) -> pd.DataFrame:
    if df is None or len(df) == 0 or "DATA_ARRECADACAO_GARE" not in df.columns:
        start = pd.Timestamp(year=ano, month=mes, day=1)
        end = start + pd.offsets.MonthEnd(0)
        return pd.DataFrame({"dia": pd.date_range(start, end, freq="D"), "valor_dia": 0.0, "n_gares": 0})
    id_col = "ID_GARE_SEFAZ" if "ID_GARE_SEFAZ" in df.columns else df.columns[0]
    daily = (
        df.dropna(subset=["DATA_ARRECADACAO_GARE"])
        .assign(dia=lambda d: d["DATA_ARRECADACAO_GARE"].dt.floor("D"))
        .groupby("dia", as_index=False)
        .agg(valor_dia=("VALOR_TOTAL_GARE", "sum"), n_gares=(id_col, "nunique"))
        .sort_values("dia")
    )
    start = pd.Timestamp(year=ano, month=mes, day=1)
    end = start + pd.offsets.MonthEnd(0)
    cal = pd.DataFrame({"dia": pd.date_range(start, end, freq="D")})
    daily = cal.merge(daily, on="dia", how="left")
    daily["valor_dia"] = daily["valor_dia"].fillna(0.0)
    daily["n_gares"] = daily["n_gares"].fillna(0).astype(int)
    return daily

def run_daily_baselines(daily: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    y = daily["valor_dia"].astype(float).values
    n = len(y)
    hold = min(7, max(3, n // 4))
    train_end = n - hold
    rows_m, preds = [], {}
    y_hold = y[train_end:]
    for name, lag in [("naive_lag1", 1), ("naive_lag7", 7)]:
        f = np.array([y[t - lag] if t - lag >= 0 else np.nan for t in range(train_end, n)], float)
        msk = ~np.isnan(f)
        rows_m.append({"modelo": name, "MAPE": mape(y_hold[msk], f[msk]),
                       "SMAPE": smape(y_hold[msk], f[msk]), "RMSE": rmse(y_hold[msk], f[msk])})
        preds[name] = f
    from sklearn.linear_model import Ridge
    feat = pd.DataFrame({"y": y})
    for lag in (1, 3, 7):
        feat[f"l{lag}"] = feat["y"].shift(lag)
    feat = feat.dropna()
    Xcols = [c for c in feat.columns if c.startswith("l")]
    fcs, acts = [], []
    for t in range(train_end, n):
        if t not in feat.index:
            fcs.append(np.nan); acts.append(y[t]); continue
        tr = feat.loc[feat.index < t]
        if len(tr) < 10:
            fcs.append(np.nan); acts.append(float(feat.loc[t, "y"])); continue
        mdl = Ridge(alpha=1.0, random_state=SEED)
        mdl.fit(tr[Xcols], tr["y"])
        fcs.append(float(mdl.predict(feat.loc[[t], Xcols])[0]))
        acts.append(float(feat.loc[t, "y"]))
    fcs = np.array(fcs, float); acts = np.array(acts, float)
    msk = ~np.isnan(fcs)
    rows_m.append({"modelo": "Ridge_lags_1_3_7", "MAPE": mape(acts[msk], fcs[msk]),
                   "SMAPE": smape(acts[msk], fcs[msk]), "RMSE": rmse(acts[msk], fcs[msk])})
    preds["Ridge_lags_1_3_7"] = fcs
    try:
        from prophet import Prophet
        dfp = pd.DataFrame({"ds": daily["dia"].iloc[:train_end], "y": y[:train_end]})
        mprop = Prophet(daily_seasonality=False, weekly_seasonality=True, yearly_seasonality=False)
        mprop.fit(dfp)
        fut = pd.DataFrame({"ds": daily["dia"].iloc[train_end:n]})
        fc = mprop.predict(fut)["yhat"].values.astype(float)
        rows_m.append({"modelo": "Prophet_daily", "MAPE": mape(y_hold, fc),
                       "SMAPE": smape(y_hold, fc), "RMSE": rmse(y_hold, fc)})
        preds["Prophet_daily"] = fc
    except Exception as e:
        print("Prophet skip", type(e).__name__, e)
    res = pd.DataFrame(rows_m).sort_values("MAPE").reset_index(drop=True)
    meta = {"hold": hold, "train_end": train_end, "n": n, "y_hold": y_hold, "preds": preds}
    return res, meta

print("helpers ok")


helpers ok


## 7. Join débito — **somente dump local** (sem API row-level)

`/v1/data/debito` já apresentou `ReadTimeout`. Tentamos apenas Parquets locais em `samples/` (`a3_debito_sample.parquet`, `toy_debito_sample*.parquet`, …). Cobertura = % de `ID_DEBITO` da janela encontrados no sample — **não** é cobertura populacional; NaN se dump ausente.


In [6]:
def load_local_debito() -> tuple[pd.DataFrame | None, str]:
    candidates = [
        "a3_debito_sample.parquet",
        "bertini_debito_sample.parquet",
        "toy_debito_sample_v2.parquet",
        "toy_debito_sample.parquet",
    ]
    frames, used = [], []
    for name in candidates:
        p = C.sample_dir() / name
        if p.exists():
            d = drop_pii(pd.read_parquet(p))
            d.columns = [c.upper() for c in d.columns]
            frames.append(d)
            used.append(name)
    if not frames:
        return None, "missing_local_debito"
    deb = pd.concat(frames, ignore_index=True)
    if "ID_DEBITO" in deb.columns:
        deb["ID_DEBITO"] = pd.to_numeric(deb["ID_DEBITO"], errors="coerce")
        deb = deb.dropna(subset=["ID_DEBITO"]).drop_duplicates("ID_DEBITO")
    print("debito local sources:", used, "| n_unique_id", len(deb) if "ID_DEBITO" in deb.columns else 0)
    print("  paths:", [public_path_label(C.sample_dir() / u) for u in used])
    return deb, "local_parquet_union"

debito_df, debito_src = load_local_debito()
print("debito_src", debito_src)
# Explicit: never call /v1/data/debito in this notebook
print("policy: NO row-level /v1/data/debito API calls")

def join_coverage(gare_df: pd.DataFrame) -> float:
    if debito_df is None or "ID_DEBITO" not in getattr(debito_df, "columns", []):
        return float("nan")
    if "ID_DEBITO" not in gare_df.columns or len(gare_df) == 0:
        return float("nan")
    ids = gare_df["ID_DEBITO"].dropna()
    if len(ids) == 0:
        return float("nan")
    return float(ids.isin(set(debito_df["ID_DEBITO"])).mean())


debito local sources: ['a3_debito_sample.parquet', 'bertini_debito_sample.parquet', 'toy_debito_sample_v2.parquet', 'toy_debito_sample.parquet'] | n_unique_id 300
  paths: ['${DUMP_ROOT}/extracao=2026-03/samples/a3_debito_sample.parquet', '${DUMP_ROOT}/extracao=2026-03/samples/bertini_debito_sample.parquet', '${DUMP_ROOT}/extracao=2026-03/samples/toy_debito_sample_v2.parquet', '${DUMP_ROOT}/extracao=2026-03/samples/toy_debito_sample.parquet']
debito_src local_parquet_union
policy: NO row-level /v1/data/debito API calls


## 8. EDA + baselines nas janelas *full*

### Como ler o histograma / barras diárias

**Histograma.** X = `VALOR_TOTAL_GARE` (R$ por guia na amostra); Y = contagem. Cauda longa é esperada — não causalizar.

**Barras diárias.** X = dia do mês-calendário; Y = soma amostral de `VALOR_TOTAL_GARE`. Dias com barra zero = sem fluxo **na amostra** (não necessariamente zero populacional).


In [7]:
forecast_results = {}  # wid -> res DataFrame
daily_store = {}

for wid in sorted(FULL_IDS, key=lambda x: int(x[1:])):
    info = loaded[wid]
    df = info["df"]
    ano, mes = info["ano"], info["mes"]
    print("=" * 60, wid, f"{ano}-{mes:02d}", "n", len(df), "src", info["source"])
    if len(df) == 0:
        print(wid, "empty — skip full EDA")
        continue
    miss = df.isna().mean().sort_values(ascending=False)
    print("missingness top:")
    display(miss.head(8).to_frame("frac_na"))
    print("valor_sum", float(df["VALOR_TOTAL_GARE"].sum()) if "VALOR_TOTAL_GARE" in df.columns else None)
    cov = join_coverage(df)
    print(f"join_coverage_local_debito = {cov if cov == cov else float('nan'):.4f}" if cov == cov else "join_coverage_local_debito = NaN")
    loaded[wid]["join_cov"] = cov

    fig_hist = px.histogram(
        df, x="VALOR_TOTAL_GARE", nbins=50,
        title=f"{wid} {ano}-{mes:02d}: distribuição VALOR_TOTAL_GARE (amostra)",
        labels={"VALOR_TOTAL_GARE": "R$"},
    )
    fig_hist.update_layout(height=400, margin=dict(t=80, l=60, r=30, b=50), title=dict(y=0.98))
    fig_hist.show()
    save_fig(fig_hist, f"gare_{wid.lower()}_hist_valor", width=900, height=400)

    daily = to_daily(df, ano, mes)
    daily_store[wid] = daily
    print("dias", len(daily), "| dias_com_fluxo", int((daily["valor_dia"] > 0).sum()))
    fig_d = go.Figure()
    fig_d.add_trace(go.Bar(x=daily["dia"], y=daily["valor_dia"], name="valor_dia"))
    fig_d.update_layout(
        title=dict(text=f"{wid} agregado diário — {ano}-{mes:02d}", y=0.98),
        height=420, yaxis_title="R$",
        margin=dict(t=80, l=60, r=30, b=50), showlegend=False,
    )
    fig_d.show()
    save_fig(fig_d, f"gare_{wid.lower()}_daily_bars", width=1000, height=420)

    res, meta = run_daily_baselines(daily)
    forecast_results[wid] = res
    loaded[wid]["best_MAPE"] = float(res.iloc[0]["MAPE"]) if len(res) else np.nan
    loaded[wid]["best_SMAPE"] = float(res.iloc[0]["SMAPE"]) if len(res) else np.nan
    loaded[wid]["best_model"] = res.iloc[0]["modelo"] if len(res) else None
    print("holdout", meta["hold"], "| best", loaded[wid]["best_model"],
          "MAPE", loaded[wid]["best_MAPE"], "SMAPE", loaded[wid]["best_SMAPE"])
    display(res)

    SHORT = {"naive_lag1": "Naive lag-1", "naive_lag7": "Naive lag-7",
             "Ridge_lags_1_3_7": "Ridge lags", "Prophet_daily": "Prophet"}
    plot_df = res.copy()
    plot_df["label"] = plot_df["modelo"].map(lambda m: SHORT.get(m, m[:18]))
    order = plot_df.sort_values("SMAPE", ascending=True)["label"].tolist()
    long = plot_df.melt(id_vars=["label"], value_vars=["MAPE", "SMAPE", "RMSE"],
                        var_name="metric", value_name="value")
    long["label"] = pd.Categorical(long["label"], categories=order, ordered=True)
    fig_cmp = px.bar(
        long, x="value", y="label", facet_col="metric", orientation="h",
        title=f"{wid} holdout diário — métricas (ordenado por SMAPE ↑)",
        labels={"value": "", "label": ""},
        category_orders={"label": order, "metric": ["MAPE", "SMAPE", "RMSE"]},
    )
    fig_cmp.update_xaxes(matches=None)
    fig_cmp.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
    fig_cmp.update_layout(height=360, margin=dict(t=90, l=120, r=40, b=40),
                          title=dict(y=0.98), showlegend=False)
    fig_cmp.show()
    save_fig(fig_cmp, f"gare_{wid.lower()}_holdout_metrics", width=1000, height=360)

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=daily["dia"].iloc[meta["train_end"]:], y=meta["y_hold"], name="Atual",
        mode="lines+markers", line=dict(width=2, color="#333"),
    ))
    for k, v in meta["preds"].items():
        fig.add_trace(go.Scatter(
            x=daily["dia"].iloc[meta["train_end"]: meta["train_end"] + len(v)], y=v,
            name=SHORT.get(k, k), mode="lines+markers",
        ))
    fig.update_layout(
        title=dict(text=f"{wid} holdout diário: atual vs baselines", y=0.98),
        height=440, yaxis_title="R$",
        margin=dict(t=100, l=60, r=40, b=50),
        legend=dict(orientation="h", yanchor="bottom", y=1.08, x=0),
    )
    fig.show()
    save_fig(fig, f"gare_{wid.lower()}_holdout_series", width=1100, height=440)


============================================================ W1 2018-01 n 4000 src parquet_cache
missingness top:


,frac_na
HONORARIOS_ADMINISTRATIVOS,1.0
ANO,0.0
MES,0.0
ID_DEBITO,0.0
CDA_COMPLETA,0.0
STATUS_AJUIZAMENTO_DEBITO,0.0
ID_GARE_SEFAZ,0.0
DATA_ARRECADACAO_GARE,0.0


valor_sum 4757247.61
join_coverage_local_debito = 0.0000


static -> mono:/projects/monitoramento/output/figures/gare/gare_w1_hist_valor.png


dias 31 | dias_com_fluxo 22


...

12:51:48 - cmdstanpy - INFO - Chain [1] done processing


holdout 7 | best naive_lag1 MAPE 0.690134334362445 SMAPE 1.0409517298984965


,modelo,MAPE,SMAPE,RMSE
0,naive_lag1,0.690134,1.040952,505860.490394
1,Ridge_lags_1_3_7,0.796270,1.321147,665535.187291
2,Prophet_daily,0.860472,1.672442,555659.236321
3,naive_lag7,0.949566,1.298130,563925.805681


static -> mono:/projects/monitoramento/output/figures/gare/gare_w6_holdout_metrics.png


static -> mono:/projects/monitoramento/output/figures/gare/gare_w6_holdout_series.png


## 9. Janelas *light* + tabela síntese cross-window

Para light: n_guias, valor_sum, missingness, join_cov (local), sem forecast (MAPE/SMAPE = NaN).


In [8]:
synth = []
for _, row in windows.iterrows():
    wid = row["id"]
    info = loaded[wid]
    df = info["df"]
    n_gares = len(df)
    valor_sum = float(df["VALOR_TOTAL_GARE"].sum()) if n_gares and "VALOR_TOTAL_GARE" in df.columns else np.nan
    na_frac = float(df.isna().mean().mean()) if n_gares else np.nan
    cov = info.get("join_cov", join_coverage(df) if n_gares else np.nan)
    entry = {
        "id": wid,
        "criterio": info["criterio"],
        "ano": info["ano"],
        "mes": info["mes"],
        "mode": info["mode"],
        "source": info["source"],
        "partial": bool(info["partial"]),
        "n_guias": n_gares,
        "valor_sum": valor_sum,
        "missingness_mean": na_frac,
        "join_coverage_local_debito": cov,
        "best_MAPE_daily": info.get("best_MAPE", np.nan),
        "best_SMAPE_daily": info.get("best_SMAPE", np.nan),
        "best_model_daily": info.get("best_model"),
    }
    synth.append(entry)
    if info["mode"] == "light" and n_gares:
        d2 = to_daily(df, info["ano"], info["mes"])
        print(wid, "light dias_com_fluxo", int((d2["valor_dia"] > 0).sum()), "valor_sum", valor_sum)

sintese = pd.DataFrame(synth)
display(sintese)
out_s = C.sample_dir() / "gare_janelas_sintese_v0.parquet"
sintese.to_parquet(out_s, index=False)
print("sintese ->", public_path_label(out_s))

# Cross-window comparison chart (n_guias + valor_sum)
fig_n = go.Figure()
fig_n.add_trace(go.Bar(x=sintese["id"], y=sintese["n_guias"], name="n_guias",
                       marker_color=["#2a9d8f" if m == "full" else "#adb5bd" for m in sintese["mode"]]))
fig_n.update_layout(title=dict(text="Síntese: n_guias por janela (verde=full, cinza=light)", y=0.98),
                    height=400, margin=dict(t=80, l=60, r=30, b=50), showlegend=False)
fig_n.show()
save_fig(fig_n, "gare_sintese_n_guias", width=1000, height=400)

full_m = sintese[sintese["mode"] == "full"].dropna(subset=["best_SMAPE_daily"])
if len(full_m):
    fig_m = px.bar(
        full_m, x="id", y="best_SMAPE_daily", color="best_model_daily",
        title="Full windows — best daily SMAPE (holdout)",
        labels={"best_SMAPE_daily": "SMAPE", "id": ""},
    )
    fig_m.update_layout(height=400, margin=dict(t=80, l=60, r=30, b=50), title=dict(y=0.98))
    fig_m.show()
    save_fig(fig_m, "gare_sintese_best_smape", width=1000, height=400)

print("n_full", int((sintese["mode"] == "full").sum()),
      "| n_light", int((sintese["mode"] == "light").sum()),
      "| timeout_incidents", len(TIMEOUT_INCIDENTS))
if TIMEOUT_INCIDENTS:
    display(pd.DataFrame(TIMEOUT_INCIDENTS))


W7 light dias_com_fluxo 20 valor_sum 4204538.87
W8 light dias_com_fluxo 20 valor_sum 570551.55
W9 light dias_com_fluxo 16 valor_sum 606903.19
W10 light dias_com_fluxo 20 valor_sum 539748.9156293256
W11 light dias_com_fluxo 2 valor_sum 749898.05
W12 light dias_com_fluxo 22 valor_sum 377173.0857366134


,id,criterio,ano,mes,mode,source,partial,n_guias,valor_sum,missingness_mean,join_coverage_local_debito,best_MAPE_daily,best_SMAPE_daily,best_model_daily
0,W1,tipico_mediana_valor_total,2018,1,full,parquet_cache,False,4000,4.757248e+06,0.071429,0.0,1.170288,0.924981,naive_lag1
1,W2,outlier_max_estoque,2020,6,full,parquet_cache,False,1000,9.573200e+05,0.071429,0.0,0.561462,0.606580,naive_lag7
2,W3,alta_arrecadacao_max,2019,12,full,api_sample,False,2000,6.622671e+06,0.071429,0.0,8.431344,1.691694,naive_lag1
3,W4,seasonal_peak_mes_12,2025,12,full,parquet_cache,False,1000,1.404175e+06,0.275000,0.0,0.536475,0.454581,naive_lag7
4,W5,mid_serie_temporal,2021,4,full,api_sample,False,2000,4.391120e+06,0.203000,0.0,0.485145,1.686218,Prophet_daily
5,W6,recente_penultimo,2026,2,full,parquet_cache,False,1000,3.904897e+06,0.191143,0.0,0.690134,1.040952,naive_lag1
6,W7,seasonal_low_mes_6,2025,6,light,api_sample,False,600,4.204539e+06,0.068095,0.0,NaN,NaN,NaN
7,W8,alta_arrec_recente_24m,2024,4,light,api_sample,False,600,5.705516e+05,0.071429,0.0,NaN,NaN,NaN
8,W9,outlier_estoque_iqr_upper,2020,5,light,api_sample,False,600,6.069032e+05,0.071429,0.0,NaN,NaN,NaN
9,W10,tipico_alt_mediana,2017,3,light,api_sample,False,600,5.397489e+05,0.109048,0.0,NaN,NaN,NaN


sintese -> ${DUMP_ROOT}/extracao=2026-03/samples/gare_janelas_sintese_v0.parquet


static -> mono:/projects/monitoramento/output/figures/gare/gare_sintese_n_guias.png


static -> mono:/projects/monitoramento/output/figures/gare/gare_sintese_best_smape.png
n_full 6 | n_light 6 | timeout_incidents 0


## 10. Conclusões (granular vs painel 119m)

- O painel mensal captura sazonalidade e ranking M0–M5 em `valor_total` agregado; as janelas GARE revelam **heterogeneidade intra-mês** (dias sem fluxo na amostra, cauda de valores de guia).
- Amostra API / cache **não** é censo: `n_guias` e `join_coverage_local_debito` são limitados por `limit`/cursor e pelo dump local de débito — não interpretar como cobertura populacional.
- Loader timeout-safe: cache-first; API com páginas curtas + retries; `partial=True` preserva páginas obtidas sem bloquear o notebook.
- **Política debito:** zero chamadas row-level a `/v1/data/debito` (histórico de ReadTimeout); join só contra Parquet local.
- Forecast diário (naive / Ridge / Prophet) nas janelas *full*; MAPE pode degradar com muitos zeros — SMAPE é mais informativo nesse grain.
- **PII:** colunas de identificação pessoal removidas do frame analítico.

### Não-afirmamos

- Não afirmamos que a amostra API reproduz o total mensal do endpoint `/serie`.
- Não afirmamos causalidade nem join completo débito↔GARE fora do sample local.
- Não inventamos métricas quando a janela ficou *light* ou vazia (NaN explícito).


## Referências

- Notebook mensal: `projects/monitoramento/notebooks/estoque_arrecadacao_eda_forecast_v0.ipynb`
- Premissa dump / plano B: `docs/planos/B_monitoramento_arrecadacao.md`, `docs/api-dump/00_premissa_dump.md`
- Hyndman, R. J., & Athanasopoulos, G. (2021). *Forecasting: Principles and Practice* (3rd ed., FPP3). https://otexts.com/fpp3/
- Makridakis, S. (1993). Accuracy measures: theoretical and practical concerns. *International Journal of Forecasting*, 9(4), 527–529. https://doi.org/10.1016/0169-2070(93)90079-3

---

**
